In [1]:
# Keep repository-relative paths valid from notebook subfolders.
from pathlib import Path
import os

os.chdir(next(
    root for root in (Path.cwd(), *Path.cwd().parents)
    if (root / "notebooks").is_dir() and (root / "requirements.txt").is_file()
))

# Resolve legacy IDX paths stored in existing CSVs without rewriting the data.
def _relocated_idx_path(value):
    text = str(value).replace("\\", "/")
    old_repo = "AI-Builders-Hackhaton-2026-Backend/"
    if old_repo in text:
        text = text.split(old_repo, 1)[1]
    old_raw = "data/idx_financial_statements/"
    if text.startswith(old_raw):
        text = "data/idx_financial/raw/" + text[len(old_raw):]
    return Path(text)

from pathlib import Path
import pandas as pd
import numpy as np
import json
import re

from openpyxl import load_workbook
from tqdm.auto import tqdm

e:\BINUS_CODING\Ai Builders Hackhaton\AI-Builders-Hackhaton-2026-Backend\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SCAN_FILE = Path(
    "data/idx_financial/inventory/idx_financial_file_scan.csv"
)

CANDIDATE_FILE = Path(
    "data/idx_financial/discovery/idx_financial_metric_candidates.csv"
)

MISSING_OUTPUT_FILE = Path(
    "data/idx_financial/discovery/idx_financial_missing_metrics.csv"
)

ENRICHED_OUTPUT_FILE = Path(
    "data/idx_financial/discovery/idx_financial_metric_candidates_enriched.csv"
)

CHECKPOINT_FILE = Path(
    "data/idx_financial/discovery/idx_missing_metrics_checkpoint.csv"
)

print("Scan exists:", SCAN_FILE.exists())
print("Candidates exists:", CANDIDATE_FILE.exists())

Scan exists: True
Candidates exists: True


In [3]:
scan_df = pd.read_csv(
    SCAN_FILE
)

candidates_df = pd.read_csv(
    CANDIDATE_FILE
)

valid_files_df = (
    scan_df[
        scan_df["scan_status"] == "VALID"
    ]
    .copy()
)

print("Valid XLSX:", len(valid_files_df))
print("Existing candidate rows:", len(candidates_df))
print("Unique tickers:", valid_files_df["ticker"].nunique())

Valid XLSX: 15821
Existing candidate rows: 173558
Unique tickers: 948


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_29048\3411384179.py:5: DtypeWarning: Columns (0: source_sheet) have mixed types. Specify dtype option on import or set low_memory=False.
  candidates_df = pd.read_csv(


In [4]:
wrong_liability_rows = candidates_df[
    candidates_df["source_label"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("total liabilities and equity")
]

print(
    "Wrong 'Total liabilities and equity' rows:",
    len(wrong_liability_rows)
)

Wrong 'Total liabilities and equity' rows: 0


In [6]:
def resolve_file_path(path_text):

    if pd.isna(path_text):
        return None

    path = _relocated_idx_path(
        str(path_text)
    )

    if path.exists():
        return path

    alternative = (
        Path.cwd() / path
    )

    if alternative.exists():
        return alternative

    return None

In [7]:
def normalize_text(value):

    if value is None:
        return ""

    text = str(value).strip().lower()

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text

In [ ]:
MISSING_METRIC_KEYWORDS = {
    "gross_profit": [
        "jumlah laba bruto",
        "total gross profit"
    ],

    "operating_cash_flow": [
        "jumlah arus kas bersih yang diperoleh dari (digunakan untuk) aktivitas operasi",
        "total net cash flows received from (used in) operating activities"
    ]
}

In [9]:
def match_missing_metric(value):

    text = normalize_text(
        value
    )

    if not text:
        return None, None

    for metric, keywords in (
        MISSING_METRIC_KEYWORDS.items()
    ):

        for keyword in keywords:

            if text == normalize_text(
                keyword
            ):
                return metric, keyword

    return None, None

In [10]:
def is_numeric_value(value):

    if value is None:
        return False

    if isinstance(
        value,
        (
            int,
            float,
            np.integer,
            np.floating
        )
    ):

        if pd.isna(value):
            return False

        return True

    return False


def extract_numeric_candidates(row_values):

    results = []

    for column_index, value in enumerate(
        row_values
    ):

        if is_numeric_value(value):

            results.append({
                "column_index":
                    column_index,

                "value":
                    value
            })

    return results

In [11]:
relevant_sheet_metrics = [
    "revenue",
    "operating_cash_flow"
]

relevant_sheets_df = (
    candidates_df[
        candidates_df["metric"]
        .isin(relevant_sheet_metrics)
    ][
        [
            "source_path",
            "source_sheet"
        ]
    ]
    .dropna()
    .drop_duplicates()
)

print(
    "Relevant file-sheet combinations:",
    len(relevant_sheets_df)
)

display(
    relevant_sheets_df.head(30)
)

Relevant file-sheet combinations: 47359


,source_path,source_sheet
6,data\idx_financial_statements\Financial_Statem...,1321000
7,data\idx_financial_statements\Financial_Statem...,1510000
10,data\idx_financial_statements\Financial_Statem...,1616000
12,data\idx_financial_statements\Financial_Statem...,1617000
20,data\idx_financial_statements\Financial_Statem...,1311000
21,data\idx_financial_statements\Financial_Statem...,1510000
24,data\idx_financial_statements\Financial_Statem...,1616000
32,data\idx_financial_statements\Financial_Statem...,1311000
33,data\idx_financial_statements\Financial_Statem...,1510000
36,data\idx_financial_statements\Financial_Statem...,1616000


In [12]:
file_to_sheets = (
    relevant_sheets_df
    .groupby("source_path")[
        "source_sheet"
    ]
    .apply(
        lambda x: list(
            dict.fromkeys(
                str(v)
                for v in x
            )
        )
    )
    .to_dict()
)

print(
    "Files with known relevant sheets:",
    len(file_to_sheets)
)

Files with known relevant sheets: 15814


In [13]:
def scan_missing_metrics_for_file(
    row,
    relevant_sheets
):

    results = []

    file_path = resolve_file_path(
        row["file_path"]
    )

    if file_path is None:
        return results

    try:

        workbook = load_workbook(
            _relocated_idx_path(file_path),
            read_only=True,
            data_only=True
        )

        available_sheets = set(
            workbook.sheetnames
        )

        sheets_to_scan = [
            str(sheet)
            for sheet in relevant_sheets
            if str(sheet)
            in available_sheets
        ]

        for sheet_name in sheets_to_scan:

            worksheet = workbook[
                sheet_name
            ]

            for row_number, cells in enumerate(
                worksheet.iter_rows(
                    values_only=True
                ),
                start=1
            ):

                for label_column, value in enumerate(
                    cells
                ):

                    if not isinstance(
                        value,
                        str
                    ):
                        continue

                    metric, matched_keyword = (
                        match_missing_metric(
                            value
                        )
                    )

                    if metric is None:
                        continue

                    numeric_candidates = (
                        extract_numeric_candidates(
                            cells
                        )
                    )

                    results.append({

                        "ticker":
                            row["ticker"],

                        "year":
                            row["year"],

                        "quarter":
                            row["quarter"],

                        "metric":
                            metric,

                        "matched_keyword":
                            matched_keyword,

                        "source_label":
                            value,

                        "source_sheet":
                            sheet_name,

                        "row_number":
                            row_number,

                        "label_column":
                            label_column,

                        "numeric_candidate_count":
                            len(
                                numeric_candidates
                            ),

                        "numeric_candidates":
                            json.dumps(
                                numeric_candidates,
                                ensure_ascii=False,
                                default=str
                            ),

                        "source_file":
                            row["file_name"],

                        "source_path":
                            row["file_path"]
                    })

        workbook.close()

    except Exception as e:

        results.append({

            "ticker":
                row["ticker"],

            "year":
                row["year"],

            "quarter":
                row["quarter"],

            "metric":
                None,

            "matched_keyword":
                None,

            "source_label":
                None,

            "source_sheet":
                None,

            "row_number":
                None,

            "label_column":
                None,

            "numeric_candidate_count":
                None,

            "numeric_candidates":
                None,

            "source_file":
                row["file_name"],

            "source_path":
                row["file_path"],

            "extraction_error":
                f"{type(e).__name__}: {e}"
        })

    return results

In [14]:
target_files_df = (
    valid_files_df[
        valid_files_df["file_path"]
        .isin(
            file_to_sheets.keys()
        )
    ]
    .copy()
)

print(
    "Files to targeted scan:",
    len(target_files_df)
)

print(
    "Compared with all valid XLSX:",
    len(valid_files_df)
)

Files to targeted scan: 15814
Compared with all valid XLSX: 15821


In [16]:
test_target_df = (
    target_files_df
    .head(5)
)

test_missing_results = []

for _, row in tqdm(
    test_target_df.iterrows(),
    total=len(test_target_df),
    desc="Testing missing metrics",
    unit="file"
):

    relevant_sheets = (
        file_to_sheets.get(
            row["file_path"],
            []
        )
    )

    results = (
        scan_missing_metrics_for_file(
            row,
            relevant_sheets
        )
    )

    test_missing_results.extend(
        results
    )


test_missing_df = pd.DataFrame(
    test_missing_results
)

display(
    test_missing_df
)

Testing missing metrics: 100%|██████████| 5/5 [00:01<00:00,  3.29file/s]


,ticker,year,quarter,metric,matched_keyword,source_label,source_sheet,row_number,label_column,numeric_candidate_count,numeric_candidates,source_file,source_path
0,ZYRX,2025,Q1,gross_profit,jumlah laba bruto,Jumlah laba bruto,1321000,8,0,2,"[{""column_index"": 1, ""value"": 9238584766}, {""c...",ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
1,ZYRX,2025,Q1,gross_profit,total gross profit,Total gross profit,1321000,8,3,2,"[{""column_index"": 1, ""value"": 9238584766}, {""c...",ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
2,ZYRX,2025,Q1,operating_cash_flow,total net cash flows received from (used in) o...,Total net cash flows received from (used in) o...,1510000,47,3,2,"[{""column_index"": 1, ""value"": -8775116976}, {""...",ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
3,ZONE,2025,Q1,gross_profit,jumlah laba bruto,Jumlah laba bruto,1311000,8,0,2,"[{""column_index"": 1, ""value"": 125876221741}, {...",ZONE_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
4,ZONE,2025,Q1,gross_profit,total gross profit,Total gross profit,1311000,8,3,2,"[{""column_index"": 1, ""value"": 125876221741}, {...",ZONE_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
5,ZONE,2025,Q1,operating_cash_flow,total net cash flows received from (used in) o...,Total net cash flows received from (used in) o...,1510000,47,3,2,"[{""column_index"": 1, ""value"": 57716995186}, {""...",ZONE_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
6,ZINC,2025,Q1,gross_profit,jumlah laba bruto,Jumlah laba bruto,1311000,8,0,2,"[{""column_index"": 1, ""value"": -19565878729}, {...",ZINC_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
7,ZINC,2025,Q1,gross_profit,total gross profit,Total gross profit,1311000,8,3,2,"[{""column_index"": 1, ""value"": -19565878729}, {...",ZINC_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
8,ZINC,2025,Q1,operating_cash_flow,total net cash flows received from (used in) o...,Total net cash flows received from (used in) o...,1510000,47,3,2,"[{""column_index"": 1, ""value"": -4855765236}, {""...",ZINC_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
9,ZATA,2025,Q1,gross_profit,jumlah laba bruto,Jumlah laba bruto,1311000,8,0,2,"[{""column_index"": 1, ""value"": 31448806432}, {""...",ZATA_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...


In [17]:
missing_results = []

total_files = len(
    target_files_df
)

CHECKPOINT_EVERY = 250


for i, (_, row) in enumerate(
    tqdm(
        target_files_df.iterrows(),
        total=total_files,
        desc="Finding missing metrics",
        unit="file"
    ),
    start=1
):

    relevant_sheets = (
        file_to_sheets.get(
            row["file_path"],
            []
        )
    )

    results = (
        scan_missing_metrics_for_file(
            row,
            relevant_sheets
        )
    )

    missing_results.extend(
        results
    )

    if (
        i % CHECKPOINT_EVERY == 0
    ):

        checkpoint_df = (
            pd.DataFrame(
                missing_results
            )
        )

        checkpoint_df.to_csv(
            CHECKPOINT_FILE,
            index=False
        )

        tqdm.write(
            f"Checkpoint saved: "
            f"{i:,} / "
            f"{total_files:,} files | "
            f"{len(checkpoint_df):,} rows"
        )

Finding missing metrics:   2%|▏         | 250/15814 [03:14<3:37:24,  1.19file/s]

Checkpoint saved: 250 / 15,814 files | 696 rows


Finding missing metrics:   3%|▎         | 500/15814 [06:36<2:35:49,  1.64file/s]

Checkpoint saved: 500 / 15,814 files | 1,394 rows


Finding missing metrics:   5%|▍         | 750/15814 [09:06<2:30:53,  1.66file/s]

...

Finding missing metrics:  98%|█████████▊| 15500/15814 [2:20:51<02:16,  2.30file/s]

Checkpoint saved: 15,500 / 15,814 files | 42,069 rows


Finding missing metrics: 100%|█████████▉| 15750/15814 [2:21:52<00:29,  2.18file/s]

Checkpoint saved: 15,750 / 15,814 files | 42,657 rows


Finding missing metrics: 100%|██████████| 15814/15814 [2:22:08<00:00,  1.85file/s]


In [18]:
missing_metrics_df = pd.DataFrame(
    missing_results
)

print(
    "New candidate rows:",
    len(missing_metrics_df)
)

display(
    missing_metrics_df.head(100)
)

New candidate rows: 42801


,ticker,year,quarter,metric,matched_keyword,source_label,source_sheet,row_number,label_column,numeric_candidate_count,numeric_candidates,source_file,source_path
0,ZYRX,2025,Q1,gross_profit,jumlah laba bruto,Jumlah laba bruto,1321000,8,0,2,"[{""column_index"": 1, ""value"": 9238584766}, {""c...",ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
1,ZYRX,2025,Q1,gross_profit,total gross profit,Total gross profit,1321000,8,3,2,"[{""column_index"": 1, ""value"": 9238584766}, {""c...",ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
2,ZYRX,2025,Q1,operating_cash_flow,total net cash flows received from (used in) o...,Total net cash flows received from (used in) o...,1510000,47,3,2,"[{""column_index"": 1, ""value"": -8775116976}, {""...",ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
3,ZONE,2025,Q1,gross_profit,jumlah laba bruto,Jumlah laba bruto,1311000,8,0,2,"[{""column_index"": 1, ""value"": 125876221741}, {...",ZONE_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
4,ZONE,2025,Q1,gross_profit,total gross profit,Total gross profit,1311000,8,3,2,"[{""column_index"": 1, ""value"": 125876221741}, {...",ZONE_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,UNTR,2025,Q1,operating_cash_flow,total net cash flows received from (used in) o...,Total net cash flows received from (used in) o...,1510000,47,3,2,"[{""column_index"": 1, ""value"": 7497846}, {""colu...",UNTR_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
96,UNTD,2025,Q1,gross_profit,jumlah laba bruto,Jumlah laba bruto,1311000,8,0,2,"[{""column_index"": 1, ""value"": 25939349082}, {""...",UNTD_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
97,UNTD,2025,Q1,gross_profit,total gross profit,Total gross profit,1311000,8,3,2,"[{""column_index"": 1, ""value"": 25939349082}, {""...",UNTD_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
98,UNTD,2025,Q1,operating_cash_flow,total net cash flows received from (used in) o...,Total net cash flows received from (used in) o...,1510000,47,3,2,"[{""column_index"": 1, ""value"": -9101200027}, {""...",UNTD_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...


In [19]:
if not missing_metrics_df.empty:

    missing_summary = (
        missing_metrics_df[
            missing_metrics_df[
                "metric"
            ].notna()
        ]
        .groupby(
            "metric"
        )
        .agg(
            matches=(
                "metric",
                "size"
            ),

            tickers=(
                "ticker",
                "nunique"
            )
        )
        .reset_index()
        .sort_values(
            "matches",
            ascending=False
        )
    )

    display(
        missing_summary
    )

,metric,matches,tickers
0,gross_profit,26994,834
1,operating_cash_flow,15807,948


In [21]:
missing_metrics_clean_df = (
    missing_metrics_df[
        missing_metrics_df[
            "metric"
        ].notna()
    ]
    .copy()
)

print(
    "Valid new metric rows:",
    len(
        missing_metrics_clean_df
    )
)

Valid new metric rows: 42801


In [ ]:
# MISSING_OUTPUT_FILE.parent.mkdir(
#     parents=True,
#     exist_ok=True
# )

# missing_metrics_clean_df.to_csv(
#     MISSING_OUTPUT_FILE,
#     index=False
# )

# print(
#     "Saved:",
#     MISSING_OUTPUT_FILE
# )

# print(
#     "Rows:",
#     len(
#         missing_metrics_clean_df
#     )
# )

NameError: name 'missing_metrics_clean_df' is not defined

In [ ]:
combined_candidates_df = pd.concat(
    [
        candidates_df,
        missing_metrics_clean_df
    ],
    ignore_index=True
)

print(
    "Before:",
    len(candidates_df)
)

print(
    "Added:",
    len(
        missing_metrics_clean_df
    )
)

print(
    "Combined:",
    len(
        combined_candidates_df
    )
)

In [ ]:
combined_candidates_df = pd.concat(
    [
        candidates_df,
        missing_metrics_clean_df
    ],
    ignore_index=True
)

print(
    "Before:",
    len(candidates_df)
)

print(
    "Added:",
    len(
        missing_metrics_clean_df
    )
)

print(
    "Combined:",
    len(
        combined_candidates_df
    )
)

In [22]:
dedupe_columns = [
    "ticker",
    "year",
    "quarter",
    "metric",
    "source_sheet",
    "row_number",
    "numeric_candidates",
    "source_file"
]

missing_metrics_clean_df = (
    missing_metrics_df
    .drop_duplicates(
        subset=dedupe_columns
    )
    .copy()
)

print(
    "Rows before dedupe:",
    len(missing_metrics_df)
)

print(
    "Rows after dedupe:",
    len(missing_metrics_clean_df)
)

Rows before dedupe: 42801
Rows after dedupe: 29304


In [23]:
clean_summary = (
    missing_metrics_clean_df
    .groupby(
        "metric"
    )
    .agg(
        matches=(
            "metric",
            "size"
        ),
        tickers=(
            "ticker",
            "nunique"
        ),
        files=(
            "source_file",
            "nunique"
        )
    )
    .reset_index()
    .sort_values(
        "matches",
        ascending=False
    )
)

display(
    clean_summary
)

,metric,matches,tickers,files
1,operating_cash_flow,15807,948,15804
0,gross_profit,13497,834,13494


In [24]:
MISSING_OUTPUT_FILE = Path(
    "data/idx_financial/discovery/idx_financial_missing_metrics.csv"
)

missing_metrics_clean_df.to_csv(
    MISSING_OUTPUT_FILE,
    index=False
)

print(
    "Saved:",
    MISSING_OUTPUT_FILE
)

print(
    "Rows:",
    len(missing_metrics_clean_df)
)

Saved: data\idx_financial_missing_metrics.csv
Rows: 29304


In [ ]:
# combined_candidates_df.to_csv(
#     ENRICHED_OUTPUT_FILE,
#     index=False
# )

# print(
#     "Saved enriched candidates to:",
#     ENRICHED_OUTPUT_FILE
# )

# print(
#     "Rows saved:",
#     len(
#         combined_candidates_df
#     )
# )

In [25]:
import pandas as pd
from pathlib import Path

OLD_FILE = Path(
    "data/idx_financial/discovery/idx_financial_metric_candidates.csv"
)

NEW_FILE = Path(
    "data/idx_financial/discovery/idx_financial_missing_metrics.csv"
)

old_df = pd.read_csv(OLD_FILE)
new_df = pd.read_csv(NEW_FILE)

print("Old rows:", len(old_df))
print("New rows:", len(new_df))

Old rows: 173558
New rows: 29304


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_29048\1669458760.py:12: DtypeWarning: Columns (0: source_sheet) have mixed types. Specify dtype option on import or set low_memory=False.
  old_df = pd.read_csv(OLD_FILE)


In [26]:
new_metric_names = (
    new_df["metric"]
    .dropna()
    .unique()
    .tolist()
)

overlap_df = old_df[
    old_df["metric"].isin(
        new_metric_names
    )
].copy()

print(
    "Rows in old CSV with new metric names:",
    len(overlap_df)
)

display(
    overlap_df.head(50)
)

Rows in old CSV with new metric names: 15807


,ticker,year,quarter,metric,matched_keyword,source_label,source_sheet,row_number,label_column,numeric_candidate_count,numeric_candidates,source_file,source_path
7,ZYRX,2025,Q1,operating_cash_flow,cash flows from operating activities,Cash flows from operating activities,1510000,6,3,0,[],ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
21,ZONE,2025,Q1,operating_cash_flow,cash flows from operating activities,Cash flows from operating activities,1510000,6,3,0,[],ZONE_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
33,ZINC,2025,Q1,operating_cash_flow,cash flows from operating activities,Cash flows from operating activities,1510000,6,3,0,[],ZINC_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
45,ZATA,2025,Q1,operating_cash_flow,cash flows from operating activities,Cash flows from operating activities,1510000,6,3,0,[],ZATA_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
55,YUPI,2025,Q1,operating_cash_flow,cash flows from operating activities,Cash flows from operating activities,1510000,6,3,0,[],YUPI_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
66,YULE,2025,Q1,operating_cash_flow,cash flows from operating activities,Cash flows from operating activities,5510000,6,3,0,[],YULE_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
76,YPAS,2025,Q1,operating_cash_flow,cash flows from operating activities,Cash flows from operating activities,1510000,6,3,0,[],YPAS_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
87,YOII,2025,Q1,operating_cash_flow,cash flows from operating activities,Cash flows from operating activities,6510000,6,3,0,[],YOII_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
97,YELO,2025,Q1,operating_cash_flow,cash flows from operating activities,Cash flows from operating activities,1510000,6,3,0,[],YELO_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
109,WTON,2025,Q1,operating_cash_flow,cash flows from operating activities,Cash flows from operating activities,1510000,6,3,0,[],WTON_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...


In [27]:
combined_df = pd.concat(
    [
        old_df,
        new_df
    ],
    ignore_index=True
)

print("Expected rows:", len(old_df) + len(new_df))
print("Combined rows:", len(combined_df))

Expected rows: 202862
Combined rows: 202862


In [28]:
dedupe_columns = [
    "ticker",
    "year",
    "quarter",
    "metric",
    "source_sheet",
    "row_number",
    "numeric_candidates",
    "source_file"
]

duplicate_count = combined_df.duplicated(
    subset=dedupe_columns,
    keep=False
).sum()

print(
    "Duplicate rows before final dedupe:",
    duplicate_count
)

Duplicate rows before final dedupe: 143324


In [29]:
combined_clean_df = (
    combined_df
    .drop_duplicates(
        subset=dedupe_columns
    )
    .copy()
)

print(
    "Rows before dedupe:",
    len(combined_df)
)

print(
    "Rows after dedupe:",
    len(combined_clean_df)
)

Rows before dedupe: 202862
Rows after dedupe: 131200


In [30]:
final_summary = (
    combined_clean_df
    .groupby("metric")
    .agg(
        rows=(
            "metric",
            "size"
        ),
        tickers=(
            "ticker",
            "nunique"
        ),
        files=(
            "source_file",
            "nunique"
        )
    )
    .reset_index()
    .sort_values(
        "rows",
        ascending=False
    )
)

display(final_summary)

,metric,rows,tickers,files
2,operating_cash_flow,31614,948,15804
3,revenue,31592,879,14398
0,cash,22857,948,15282
4,total_assets,15820,948,15817
5,total_liabilities,15820,948,15817
1,gross_profit,13497,834,13494


In [31]:
old_ocf_check = (
    old_df[
        old_df["metric"].eq(
            "operating_cash_flow"
        )
    ]
    .groupby(
        "numeric_candidate_count"
    )
    .size()
    .reset_index(
        name="rows"
    )
)

display(old_ocf_check)

,numeric_candidate_count,rows
0,0,15807


In [32]:
old_df_clean = old_df[
    ~(
        old_df["metric"].eq(
            "operating_cash_flow"
        )
        &
        old_df["numeric_candidate_count"].eq(0)
    )
].copy()

print(
    "Old rows before:",
    len(old_df)
)

print(
    "Old rows after removing empty OCF:",
    len(old_df_clean)
)

Old rows before: 173558
Old rows after removing empty OCF: 157751


In [33]:
print(
    old_df_clean["metric"]
    .value_counts()
)

metric
revenue              48757
cash                 45714
total_assets         31640
total_liabilities    31640
Name: count, dtype: int64


In [34]:
combined_df = pd.concat(
    [
        old_df_clean,
        new_df
    ],
    ignore_index=True
)

print(
    "Combined rows:",
    len(combined_df)
)

Combined rows: 187055


In [ ]:
OUTPUT_FILE = Path(
    "data/idx_financial/discovery/idx_financial_metric_candidates_enriched.csv"
)

combined_clean_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print(
    "Saved:",
    OUTPUT_FILE
)

print(
    "Final rows:",
    len(combined_clean_df)
)

In [35]:
dedupe_columns = [
    "ticker",
    "year",
    "quarter",
    "metric",
    "source_sheet",
    "row_number",
    "numeric_candidates",
    "source_file"
]

combined_clean_df = (
    combined_df
    .drop_duplicates(
        subset=dedupe_columns
    )
    .copy()
)

print(
    "Rows before dedupe:",
    len(combined_df)
)

print(
    "Rows after dedupe:",
    len(combined_clean_df)
)

Rows before dedupe: 187055
Rows after dedupe: 115393


In [36]:
final_summary = (
    combined_clean_df
    .groupby("metric")
    .agg(
        rows=("metric", "size"),
        tickers=("ticker", "nunique"),
        files=("source_file", "nunique")
    )
    .reset_index()
    .sort_values(
        "rows",
        ascending=False
    )
)

display(final_summary)

,metric,rows,tickers,files
3,revenue,31592,879,14398
0,cash,22857,948,15282
5,total_liabilities,15820,948,15817
4,total_assets,15820,948,15817
2,operating_cash_flow,15807,948,15804
1,gross_profit,13497,834,13494


In [37]:
from pathlib import Path

OUTPUT_FILE = Path(
    "data/idx_financial/discovery/idx_financial_metric_candidates_enriched.csv"
)

combined_clean_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print(
    "Saved:",
    OUTPUT_FILE
)

print(
    "Final rows:",
    len(combined_clean_df)
)

Saved: data\idx_financial_metric_candidates_enriched.csv
Final rows: 115393
